# Etape 4 - Variations stylistiques et signal de genre

Ce notebook examine si le sexe des candidats est associe a des differences stylistiques dans les professions de foi. Pipeline : extraction de features stylistiques enrichies (Argamon & Koppel 2003, Pennebaker LIWC, Sarawgi et al. 2011), tests non-parametriques, regression OLS avec controle du parti, classification supervisee.

## 0. Imports

In [ ]:
import sys
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from scipy import stats
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

DATA_DIR    = Path("../data")
PROC_DIR    = DATA_DIR / "processed"
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)
print("Librairies chargees.")

## 1. Chargement des donnees

In [ ]:
import zipfile, sys
sys.path.append("../src")
from function import clean_text_for_style, classify_candidate_support, msttr

# ── Métadonnées ──────────────────────────────────────────────────────────────
meta = pd.read_csv(DATA_DIR / "meta_archelect_1993.csv")

# ── Textes bruts depuis le ZIP ───────────────────────────────────────────────
raw_texts = {}
with zipfile.ZipFile(DATA_DIR / "legislatives_1993.zip") as z:
    for name in z.namelist():
        if name.endswith(".txt"):
            doc_id = name.split("/")[-1].replace(".txt", "")
            with z.open(name) as f:
                raw_texts[doc_id] = f.read().decode("utf-8", errors="replace")

print(f"Textes bruts chargés : {len(raw_texts):,}")

# ── Construction du dataframe ────────────────────────────────────────────────
df_raw = pd.DataFrame([
    {"id": doc_id, "raw_text": text}
    for doc_id, text in raw_texts.items()
])

df = df_raw.merge(meta, on="id", how="inner")
print(f"Après jointure métadonnées : {len(df):,}")

# ── Famille partisane (recalculée ici, sans dépendance externe) ──────────────
codage = df["titulaire-soutien"].apply(classify_candidate_support)
df = pd.concat([df, codage], axis=1)
df["titulaire-soutien-simplifie"] = df["famille_partisane"]

# ── Filtrage sur le genre renseigné ──────────────────────────────────────────
df_gendered = df[df["titulaire-sexe"].isin(["homme", "femme"])].copy().reset_index(drop=True)

# ── Nettoyage stylométrique sur texte brut ───────────────────────────────────
print("Application de clean_text_for_style...")
df_gendered["text_style"] = df_gendered["raw_text"].apply(clean_text_for_style)

n_h = (df_gendered["titulaire-sexe"] == "homme").sum()
n_f = (df_gendered["titulaire-sexe"] == "femme").sum()
print(f"Corpus avec genre : {len(df_gendered):,}")
print(f"  Hommes : {n_h:,} ({n_h/(n_h+n_f)*100:.1f}%)")
print(f"  Femmes : {n_f:,} ({n_f/(n_h+n_f)*100:.1f}%)")


## 2. Extraction des features stylistiques

Features issues de la litterature sur la stylometrie de genre : mots fonctionnels, hedging, vocabulaire social vs economique, imperatifs, passif, profondeur syntaxique, nominalisations, registre d'adresse, bigrammes POS.

In [ ]:
try:
    nlp = spacy.load("fr_core_news_md")
except OSError:
    nlp = spacy.load("fr_core_news_sm")

PRON_1SG   = {"je", "j", "me", "moi", "mon", "ma", "mes"}
PRON_1PL   = {"nous", "notre", "nos", "on"}
MODAUX     = {"devoir", "pouvoir", "falloir", "vouloir", "savoir", "oser"}
VOUS_SET   = {"vous", "votre", "vos"}
TU_SET     = {"tu", "te", "ton", "ta", "tes", "toi"}
HEDGES     = {"peut-etre", "probablement", "sans doute", "eventuellement",
              "possiblement", "apparemment", "sembler", "croire", "supposer"}
CERTAINTY  = {"certainement", "absolument", "evidemment", "clairement",
              "forcement", "necessairement", "incontestablement", "assurement"}
SOCIAL_W   = {"famille", "enfant", "ecole", "sante", "education", "solidarite",
              "pauvrete", "logement", "handicap", "protection", "femme", "jeune"}
ECONOMIC_W = {"entreprise", "emploi", "fiscal", "impot", "croissance",
              "investissement", "budget", "dette", "economie", "competitivite"}
NOMI_SUFF  = ("tion", "ment", "ite", "eur", "esse", "isme", "ance", "ence")

def dep_depth(token):
    d, t = 0, token
    while t.head != t:
        t = t.head
        d += 1
    return d

def extract_style_features(text):
    doc       = nlp(text[:8000])
    tok_alpha = [t for t in doc if t.is_alpha]
    n_tokens  = len(tok_alpha) or 1
    lemmas    = [t.lemma_.lower() for t in tok_alpha]
    pos_list  = [t.pos_ for t in tok_alpha]
    sentences = [s for s in doc.sents if any(t.is_alpha for t in s)]
    sent_lens = [len([t for t in s if t.is_alpha]) for s in sentences]
    word_lens = [len(t.text) for t in tok_alpha]

    # ── MSTTR calculé directement sur les lemmes spaCy (fenêtre 100) ──
    msttr_score = msttr(lemmas, window=100)

    feat = {
        "mean_sent_len" : float(np.mean(sent_lens)) if sent_lens else 0.0,
        "mean_word_len" : float(np.mean(word_lens)) if word_lens else 0.0,
        "msttr"         : msttr_score,
        "pron_1sg_rate" : sum(1 for l in lemmas if l in PRON_1SG) / n_tokens * 100,
        "pron_1pl_rate" : sum(1 for l in lemmas if l in PRON_1PL) / n_tokens * 100,
        "modal_rate"    : sum(1 for l in lemmas if l in MODAUX)   / n_tokens * 100,
        "adj_rate"      : pos_list.count("ADJ") / n_tokens * 100,
        "det_rate"      : pos_list.count("DET")   / n_tokens * 100,
        "prep_rate"     : pos_list.count("ADP")   / n_tokens * 100,
        "cconj_rate"    : pos_list.count("CCONJ") / n_tokens * 100,
        "sconj_rate"    : pos_list.count("SCONJ") / n_tokens * 100,
        "hedge_rate"    : sum(1 for l in lemmas if l in HEDGES)    / n_tokens * 100,
        "certainty_rate": sum(1 for l in lemmas if l in CERTAINTY) / n_tokens * 100,
        "social_rate"   : sum(1 for l in lemmas if l in SOCIAL_W)   / n_tokens * 100,
        "economic_rate" : sum(1 for l in lemmas if l in ECONOMIC_W) / n_tokens * 100,
        "imp_rate"      : sum(1 for t in tok_alpha if "Imp" in t.morph.get("Mood", [])) / n_tokens * 100,
        "passive_rate"  : sum(1 for t in doc if t.dep_ == "auxpass") / n_tokens * 100,
        "mean_dep_depth": float(np.mean([dep_depth(t) for t in tok_alpha])) if tok_alpha else 0.0,
        "nomi_rate"     : sum(1 for t in tok_alpha if t.pos_ == "NOUN"
                              and t.lemma_.lower().endswith(NOMI_SUFF)) / n_tokens * 100,
        "vous_rate"     : sum(1 for l in lemmas if l in VOUS_SET) / n_tokens * 100,
        "tu_rate"       : sum(1 for l in lemmas if l in TU_SET)   / n_tokens * 100,
    }
    pos_bigrams = [f"{pos_list[i]}_{pos_list[i+1]}" for i in range(len(pos_list)-1)]
    feat.update({f"bg_{k}": v / n_tokens * 100 for k, v in Counter(pos_bigrams).items()})
    return feat

print("Extraction en cours...")
style_rows = [extract_style_features(t) for t in tqdm(df_gendered["text_style"])]
df_style   = pd.DataFrame(style_rows).fillna(0)

BASE_FEATURES   = [c for c in df_style.columns if not c.startswith("bg_")]
BIGRAM_FEATURES = [c for c in df_style.columns if c.startswith("bg_")]
top20_bigrams   = df_style[BIGRAM_FEATURES].sum().sort_values(ascending=False).head(20).index.tolist()
STYLE_FEATURES  = BASE_FEATURES + top20_bigrams

for col in STYLE_FEATURES:
    df_gendered[col] = df_style[col].values

print(f"Features extraites : {len(STYLE_FEATURES)} ({len(BASE_FEATURES)} base + {len(top20_bigrams)} bigrammes POS)")


## 3. Analyse descriptive

In [ ]:
desc = df_gendered.groupby("titulaire-sexe")[BASE_FEATURES].agg(["mean","std"]).round(3)
display(desc.T)

### 3.1 Violin plots

In [ ]:
n_feat = len(BASE_FEATURES)
ncols  = 4
nrows  = (n_feat + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows*4))
axes = axes.flatten()
palette = {"homme": "#4C8BE2", "femme": "#E24C8B"}
for ax, feat in zip(axes, BASE_FEATURES):
    sns.violinplot(data=df_gendered, x="titulaire-sexe", y=feat,
                   palette=palette, ax=ax, inner="quartile", cut=0)
    ax.set_title(feat, fontsize=9); ax.set_xlabel("")
for ax in axes[n_feat:]: ax.set_visible(False)
plt.suptitle("Distributions stylistiques par sexe", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_style_violins.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 Tests de Mann-Whitney U avec correction de Bonferroni

In [ ]:
alpha_bonf = 0.05 / len(STYLE_FEATURES)
results = []
for feat in STYLE_FEATURES:
    h = df_gendered[df_gendered["titulaire-sexe"] == "homme"][feat].dropna()
    f = df_gendered[df_gendered["titulaire-sexe"] == "femme"][feat].dropna()
    stat, p = stats.mannwhitneyu(h, f, alternative="two-sided")
    n = len(h) + len(f)
    z = (stat - len(h)*len(f)/2) / np.sqrt(len(h)*len(f)*(n+1)/12)
    r = abs(z) / np.sqrt(n)
    results.append({"feature": feat, "mean_H": h.mean(), "mean_F": f.mean(),
                    "diff_F_H": f.mean()-h.mean(), "p_value": p,
                    "effect_r": r, "significant": p < alpha_bonf})
df_tests = pd.DataFrame(results).sort_values("p_value")
n_sig = df_tests["significant"].sum()
print(f"Seuil Bonferroni : {alpha_bonf:.5f}")
print(f"Features significatives : {n_sig} / {len(STYLE_FEATURES)}")
display(df_tests.head(15).round(5))

### 3.3 Regression OLS avec controle du parti

In [ ]:
df_reg = df_gendered.copy()
df_reg["sexe_bin"] = (df_reg["titulaire-sexe"] == "femme").astype(int)
df_reg.columns = df_reg.columns.str.replace(r"[^a-zA-Z0-9_]", "_", regex=True)
reg_results = []
for feat in STYLE_FEATURES:
    feat_safe = feat.replace("-","_")
    try:
        model = smf.ols(f"{feat_safe} ~ sexe_bin + C(titulaire_soutien_simplifie)", data=df_reg).fit()
        coef  = model.params["sexe_bin"]
        pval  = model.pvalues["sexe_bin"]
        ci_l, ci_h = model.conf_int().loc["sexe_bin"]
        reg_results.append({"feature": feat, "coef_sexe": coef,
                            "ci_low": ci_l, "ci_high": ci_h,
                            "p_value": pval, "r2": model.rsquared,
                            "significant": pval < 0.05})
    except: pass
df_reg_res = pd.DataFrame(reg_results).sort_values("p_value")
n_sig_ols  = df_reg_res["significant"].sum()
print(f"Features significatives apres controle parti : {n_sig_ols} / {len(STYLE_FEATURES)}")
display(df_reg_res.head(15).round(4))

In [ ]:
top20 = df_reg_res.head(20)
colors = ["#E24C8B" if s else "#888888" for s in top20["significant"]]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20["feature"], top20["coef_sexe"], color=colors, alpha=0.8)
ax.errorbar(top20["coef_sexe"], top20["feature"],
            xerr=[top20["coef_sexe"]-top20["ci_low"], top20["ci_high"]-top20["coef_sexe"]],
            fmt="none", color="black", capsize=4, lw=1.5)
ax.axvline(0, color="black", lw=0.8, linestyle="--")
ax.set_xlabel("Coefficient (femme vs homme, controle parti)")
ax.set_title("Top 20 features — Effet du sexe (rose = p<0.05)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_style_regression_coefs.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Classification supervisee

In [ ]:
X = df_gendered[STYLE_FEATURES].values
y = (df_gendered["titulaire-sexe"] == "femme").astype(int).values
clf = Pipeline([("scaler", StandardScaler()),
                ("lr", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Validation croisee (5 folds) :")
for metric in ("accuracy", "f1_macro", "roc_auc"):
    scores = cross_val_score(clf, X, y, cv=cv, scoring=metric)
    print(f"  {metric:15s} : {scores.mean():.4f} +/- {scores.std():.4f}")
baseline_acc = (y == 0).mean()
print(f"  Baseline (majorite) : {baseline_acc:.4f}")

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
clf.fit(X_tr, y_tr); y_pred = clf.predict(X_te)
print(classification_report(y_te, y_pred, target_names=["homme", "femme"]))
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(y_te, y_pred, display_labels=["homme","femme"],
                                        colorbar=False, ax=ax)
ax.set_title("Matrice de confusion (test 20%)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
coefs     = clf.named_steps["lr"].coef_[0]
feat_imp  = pd.Series(coefs, index=STYLE_FEATURES)
top20_imp = feat_imp.abs().sort_values(ascending=False).head(20)
colors_imp = ["#E24C8B" if coefs[STYLE_FEATURES.index(f)] > 0 else "#4C8BE2" for f in top20_imp.index]
fig, ax = plt.subplots(figsize=(9,6))
top20_imp.plot.barh(ax=ax, color=colors_imp)
ax.set_xlabel("|Coefficient| (rose=femme, bleu=homme)")
ax.set_title("Top 20 features par importance (regression logistique)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Synthese

In [ ]:
sig_mw   = df_tests[df_tests["significant"]]["feature"].tolist()
sig_ols  = df_reg_res[df_reg_res["significant"]]["feature"].tolist()
sig_both = sorted(set(sig_mw) & set(sig_ols))
print("=" * 60)
print("SYNTHESE - Analyse stylistique par genre")
print("=" * 60)
print(f"Features Mann-Whitney significatives  : {len(sig_mw)} -> {sig_mw}")
print(f"Features OLS significatives           : {len(sig_ols)} -> {sig_ols}")
print(f"Features robustes aux deux tests      : {sig_both}")
print(f"Baseline (majorite)                   : {baseline_acc:.3f}")

## 6. Sauvegarde

In [ ]:
STYLE_COLS = ["id", "titulaire-sexe", "titulaire-soutien-simplifie"] + STYLE_FEATURES
df_gendered[STYLE_COLS].to_csv(PROC_DIR / "style_features_1993.csv", index=False)
print("Sauvegarde :", PROC_DIR / "style_features_1993.csv")
print(f"Shape : {df_gendered[STYLE_COLS].shape}")